<a href="https://colab.research.google.com/github/JoGabTasca/NLP/blob/main/PLN_aula07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from wordcloud import WordCloud
import nltk
import matplotlib.pyplot as plt
import re
import spacy

In [ ]:
page = requests.get("https://www.vagalume.com.br/linkin-park/")
soup = BeautifulSoup(page.content, 'html.parser')
lista_alfabetica = BeautifulSoup(str(soup.find_all(id = "alfabetMusicList")), 'html.parser')
a_tag = lista_alfabetica.find_all('a')

musicas = []
for a in a_tag:
    nome_musica = a.text
    if not(nome_musica == 'TRADUÇÃO' or nome_musica == ''):
        link_musica = a['href']
        musicas.append([nome_musica, link_musica])

for i in range(len(musicas)):
    link = "https://www.vagalume.com.br" + str(musicas[i][1])
    page = requests.get(link)
    soup = BeautifulSoup(page.content, 'html.parser')
    h3_tag = soup.find_all('h3')
    if len(h3_tag) != 0:
        album = h3_tag[0].text
    else:
        album = ''
    lyrics = soup.find_all(id = 'lyrics')
    lyrics = str(lyrics[0])
    lyrics = lyrics.replace('<div id="lyrics">', '')
    lyrics = lyrics.replace('<div data-plugin="googleTranslate" id="lyrics">', '')
    lyrics = lyrics.replace('<br/>', ' ')
    lyrics = lyrics.replace("\'","'")
    lyrics = lyrics.replace('</div>', '')
    musicas[i].append(album)
    musicas[i].append(lyrics)

musicas = pd.DataFrame(musicas, columns=['Nome da Música', 'link', 'album', 'letra'])

In [ ]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 41.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
spc_en = spacy.load('en_core_web_sm')

In [ ]:
nltk.download('stopwords')
from nltk.corpus import stopwords as nltk_stopwords

# Update the stopwords set to use English stopwords from NLTK
stopwords = set(nltk_stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
def limpar_texto(texto):
    # Encontra palavars completas, incluindo as acentuadas e desconsiderando pontuações.
    letras =  re.findall(r'\b[A-zÀ-úü]+\b', texto.lower())

    # Aplicando as stopwors
    meaningful_words = [w for w in letras if w not in stopwords]
    meaningful_words_string = " ".join(meaningful_words)

    # Instanciando o objeto Spacy
    spc_letras =  spc_en(meaningful_words_string)

    # Lematização e Tokenização usando a Spacy
    # Extrai a forma base (lema) de uma palavra somente se essa palavra
    # for identificada como um verbo.
    tokens = [token.lemma_ if token.pos_=='VERB' else str(token) for token in spc_letras]

    return tokens

In [ ]:
musicas['letra_limpa'] = musicas['letra'].apply(lambda x: ' '.join(limpar_texto(x)))

# Display the updated DataFrame to show the new column
musicas.head()

,Nome da Música,link,album,letra,letra_limpa
0,1stp Klosr,/linkin-park/1stp-klosr.html,Reanimation,Break I'm about to break This room to breathe ...,break break room breathe can not take anymore ...
1,26 Lettaz In da Alphabet,/linkin-park/26-lettaz-in-da-alphabet.html,,"Man, the other day I was trying to figure out ...",man day try figure many letters alphabet take ...
2,A Light That Never Comes (Feat. Steve Aoki),/linkin-park/a-light-that-never-comes-with-ste...,Recharged,"Nah, you don't know me Lightning above and a f...",nah know lightning fire can not catch can not ...
3,A Line In The Sand,/linkin-park/a-line-in-the-sand.html,The Hunting Party,Today We stood on the wall We laughed at the s...,today stand wall laugh sun laugh guns laugh te...
4,A Place For My Head,/linkin-park/a-place-for-my-head.html,Hybrid Theory,I watch how the moon sits in the sky in the da...,watch moon sit sky dark night shin light sun s...


In [ ]:
from nltk import word_tokenize
# nltk.download('punkt') # Already downloaded and not the source of the error

# Combine all cleaned lyrics into a single string for vocabulary extraction
all_cleaned_lyrics = ' '.join(musicas['letra_limpa'])

# Tokenize the combined cleaned lyrics using split() instead of word_tokenize
tokens = all_cleaned_lyrics.split()

# Create a vocabulary list of unique tokens
Vocab = []
for token in tokens:
    if token not in Vocab:
        Vocab.append(token)

print(f"Total unique words in vocabulary: {len(Vocab)}")
print("First 20 words in vocabulary:", Vocab[:20])

Total unique words in vocabulary: 3754
First 20 words in vocabulary: ['break', 'room', 'breathe', 'can', 'not', 'take', 'anymore', 'say', 'everything', 'words', 'make', 'sense', 'find', 'bliss', 'ignorance', 'less', 'hear', 'anyway', 'answers', 'clear']


In [ ]:
def calculaTFIDF(tf_bow, idfs):
    tfidf = {}
    for palavra in tf_bow:
        tf = tf_bow[palavra]
        idf = idfs[palavra]
        tfidf[palavra] = tf*idf
    return(tfidf)

In [ ]:
from collections import Counter

def calculate_tf(document):
    # document is a string of space-separated words (letra_limpa)
    tokens = document.split()
    tf_bow = Counter(tokens)
    return tf_bow

# Apply the TF calculation to each song's cleaned lyrics
musicas['tf_bow'] = musicas['letra_limpa'].apply(calculate_tf)

print("Term Frequency (TF) for the first song:")
print(musicas['tf_bow'].iloc[0])

Term Frequency (TF) for the first song:
Counter({'break': 14, 'say': 9, 'room': 7, 'breathe': 7, 'everything': 7, 'need': 6, 'little': 6, 'find': 5, 'shut': 5, 'pour': 4, 'one': 4, 'step': 4, 'closer': 4, 'edge': 4, 'take': 3, 'like': 3, 'blood': 3, 'can': 2, 'not': 2, 'make': 2, 'sense': 2, 'bliss': 2, 'ignorance': 2, 'less': 2, 'talk': 2, 'cause': 2, 'anymore': 1, 'words': 1, 'hear': 1, 'anyway': 1, 'answers': 1, 'clear': 1, 'wish': 1, 'could': 1, 'way': 1, 'disappear': 1, 'thoughts': 1, 'nothing': 1, 'seem': 1, 'go': 1, 'away': 1, 'places': 1, 'feel': 1, 'torn': 1, 'body': 1, 'flesh': 1, 'peels': 1, 'ride': 1, 'cut': 1, 'wait': 1, 'alone': 1, 'resist': 1, 'feeling': 1, 'hate': 1, 'never': 1, 'miss': 1, 'please': 1, 'someone': 1, 'give': 1, 'reason': 1, 'rip': 1, 'face': 1})


In [ ]:
import math

def calculate_idf(documents, vocab):
    idf = {}
    N = len(documents) # Total number of documents
    for word in vocab:
        doc_count = 0 # Number of documents containing the word
        for doc_tf_bow in documents:
            if word in doc_tf_bow:
                doc_count += 1
        # Add 1 to the denominator to avoid division by zero for unseen words
        idf[word] = math.log(N / (doc_count + 1))
    return idf

# Calculate IDF using the 'tf_bow' column and the 'Vocab'
idfs = calculate_idf(musicas['tf_bow'], Vocab)

print("Inverse Document Frequency (IDF) for selected words:")
print({word: idfs[word] for word in list(Vocab)[:10]})
print(f"Total IDF values calculated: {len(idfs)}")

Inverse Document Frequency (IDF) for selected words:
{'break': 1.5591371739593014, 'room': 2.9644797300498866, 'breathe': 3.1315338147130527, 'can': 2.8903717578961645, 'not': 2.8903717578961645, 'take': 0.9444616088408515, 'anymore': 3.044522437723423, 'say': 0.7844969591481732, 'everything': 1.8158570208071154, 'words': 2.063693184711697}
Total IDF values calculated: 3754


In [ ]:
# Apply the calculaTFIDF function to each song
musicas['tfidf'] = musicas['tf_bow'].apply(lambda tf_bow: calculaTFIDF(tf_bow, idfs))

print("TF-IDF for the first song (showing first 5 words with non-zero TF-IDF):")
first_song_tfidf = {k: v for k, v in musicas['tfidf'].iloc[0].items() if v != 0.0}
# Sort for consistent output and display top 5
print(dict(sorted(first_song_tfidf.items(), key=lambda item: item[1], reverse=True)[:5]))

TF-IDF for the first song (showing first 5 words with non-zero TF-IDF):
{'breathe': 21.92073670299137, 'break': 21.82792043543022, 'room': 20.751358110349205, 'closer': 14.950678473133474, 'little': 14.363609229493642}


In [ ]:
# Display the DataFrame head with the new 'tfidf' column
display(musicas[['Nome da Música', 'letra_limpa', 'tfidf']])

,Nome da Música,letra_limpa,tfidf
0,1stp Klosr,break break room breathe can not take anymore ...,"{'break': 21.82792043543022, 'room': 20.751358..."
1,26 Lettaz In da Alphabet,man day try figure many letters alphabet take ...,"{'man': 5.169980216689965, 'day': 3.8370223497..."
2,A Light That Never Comes (Feat. Steve Aoki),nah know lightning fire can not catch can not ...,"{'nah': 8.861633597686627, 'know': 2.276233389..."
3,A Line In The Sand,today stand wall laugh sun laugh guns laugh te...,"{'today': 9.939626599152001, 'stand': 4.065843..."
4,A Place For My Head,watch moon sit sky dark night shin light sun s...,"{'watch': 1.840549633397487, 'moon': 11.759973..."
...,...,...,...
247,Wretches and Kings,time operation machine become odious make sick...,"{'time': 1.2809338454620642, 'operation': 9.67..."
248,Wth> You,come wake dream today cold static put cold fee...,"{'come': 4.065125452847358, 'wake': 8.46413665..."
249,X-ecutioner Style,top shut shut talk shut fun let try something ...,"{'top': 2.8213788864092133, 'shut': 11.2855155..."
250,Yo (Minutes To Midnight Demo),instrumental linkin park underground,"{'instrumental': 1.8918429277850375, 'linkin':..."
